In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../dataset/transactions.csv", parse_dates=["date"])
df.shape

(20040, 8)

In [2]:
def build_features(df):
    out = df.copy()
    typical = out.groupby(["user_id", "category"])["amount"].transform("median")
    out["amount_ratio"] = out["amount"] / typical
    out["log_ratio"] = np.log(out["amount_ratio"])
    out["is_night"] = out["hour"].between(0, 5).astype(int)
    out["is_repeat"] = (
        out.groupby(["user_id", "date", "description", "amount"]).cumcount() > 0
    ).astype(int)
    return out

FEATURES = ["log_ratio", "hour", "is_night", "is_repeat"]

feat = build_features(df)
feat[["user_id", "category", "amount", "amount_ratio", "log_ratio", "hour", "is_night", "is_repeat"]].head(10).round(2)

,user_id,category,amount,amount_ratio,log_ratio,hour,is_night,is_repeat
0,U01,Groceries,1264.0,1.28,0.25,11,0,0
1,U01,Groceries,640.0,0.65,-0.43,20,0,0
2,U01,Rent,23000.0,1.00,0.00,10,0,0
3,U01,Food,595.0,1.59,0.46,11,0,0
4,U01,Transport,334.0,1.53,0.42,13,0,0
5,U01,Food,310.0,0.83,-0.19,14,0,0
6,U01,Food,362.0,0.97,-0.03,19,0,0
7,U01,Shopping,5666.0,3.40,1.22,16,0,0
8,U01,Transport,211.0,0.97,-0.03,13,0,0
9,U01,Transport,131.0,0.60,-0.51,14,0,0


In [3]:
feat.groupby("anomaly_type")[FEATURES].mean().round(2)

,log_ratio,hour,is_night,is_repeat
anomaly_type,,,,
duplicate,-0.05,14.84,0.0,1.0
none,-0.01,14.44,0.0,0.0
odd_hour,1.66,2.35,1.0,0.0
spike,2.30,14.13,0.0,0.0


In [4]:
pd.crosstab(feat["is_repeat"], feat["is_anomaly"], rownames=["is_repeat"], colnames=["is_anomaly"])

is_anomaly,0,1
is_repeat,,
0,19740,223
1,1,76


In [5]:
print(feat.groupby("anomaly_type")["log_ratio"].describe().round(2))

                count  mean   std   min   25%   50%   75%   max
anomaly_type                                                   
duplicate        76.0 -0.05  0.42 -1.21 -0.25 -0.03  0.22  1.18
none          19741.0 -0.01  0.54 -2.77 -0.35  0.00  0.34  2.63
odd_hour         74.0  1.66  0.58  0.25  1.35  1.68  2.09  3.09
spike           149.0  2.30  0.63  0.77  1.84  2.39  2.69  3.85


In [6]:
from sklearn.ensemble import IsolationForest

X = feat[FEATURES]
iso = IsolationForest(n_estimators=200, contamination=0.015, random_state=42)
iso.fit(X)

feat["pred"] = (iso.predict(X) == -1).astype(int)
feat["score"] = -iso.decision_function(X)   # higher = more anomalous
print("Flagged:", feat["pred"].sum(), "of", len(feat))

Flagged: 301 of 20040


In [7]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(feat["is_anomaly"], feat["pred"], digits=3))
print(confusion_matrix(feat["is_anomaly"], feat["pred"]))

              precision    recall  f1-score   support

           0      0.994     0.994     0.994     19741
           1      0.628     0.632     0.630       299

    accuracy                          0.989     20040
   macro avg      0.811     0.813     0.812     20040
weighted avg      0.989     0.989     0.989     20040

[[19629   112]
 [  110   189]]


In [8]:
feat[feat["is_anomaly"] == 1].groupby("anomaly_type")["pred"].mean().round(2)

anomaly_type
duplicate    0.26
odd_hour     1.00
spike        0.64
Name: pred, dtype: float64

In [9]:
from sklearn.metrics import roc_auc_score
print("AUC:", round(roc_auc_score(feat["is_anomaly"], feat["score"]), 4))

AUC: 0.9899


In [10]:
cols = ["user_id", "date", "hour", "description", "category", "amount", "anomaly_type", "score"]
feat.sort_values("score", ascending=False)[cols].head(15).round(2)

C:\Users\hp\AppData\Local\Temp\ipykernel_13584\3924728781.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  feat.sort_values("score", ascending=False)[cols].head(15).round(2)


,user_id,date,hour,description,category,amount,anomaly_type,score
7237,U11,2026-10-04,1,Indian Oil,Transport,3850.0,odd_hour,0.19
5437,U09,2026-01-23,1,Transfer to Friend,Other,10444.0,odd_hour,0.18
18774,U29,2025-12-19,1,ATM Withdrawal,Other,9011.0,odd_hour,0.18
2289,U04,2026-05-08,1,Rapido,Transport,3283.0,odd_hour,0.18
5022,U08,2026-06-08,3,Rapido,Transport,2952.0,odd_hour,0.18
14089,U21,2026-10-29,3,Mutual Fund SIP,Other,6327.0,odd_hour,0.18
9571,U15,2026-03-28,3,Rapido,Transport,2512.0,odd_hour,0.18
17353,U26,2026-10-21,2,Rapido,Transport,1417.0,odd_hour,0.18
1785,U03,2026-08-04,3,Chaayos,Food,3484.0,odd_hour,0.18
5659,U09,2026-05-08,1,Indian Oil,Transport,1308.0,odd_hour,0.18


In [11]:
ML_FEATURES = ["log_ratio", "hour", "is_night"]   # is_repeat is now a rule, not an ML feature

iso2 = IsolationForest(n_estimators=200, contamination=0.015, random_state=42)
iso2.fit(feat[ML_FEATURES])

feat["flag_ml"] = (iso2.predict(feat[ML_FEATURES]) == -1).astype(int)
feat["flag_rule"] = feat["is_repeat"]
feat["flag_hybrid"] = ((feat["flag_ml"] == 1) | (feat["flag_rule"] == 1)).astype(int)

print("Flagged by ML:    ", feat["flag_ml"].sum())
print("Flagged by rule:  ", feat["flag_rule"].sum())
print("Flagged in total: ", feat["flag_hybrid"].sum())

Flagged by ML:     301
Flagged by rule:   77
Flagged in total:  378


In [12]:
print(classification_report(feat["is_anomaly"], feat["flag_hybrid"], digits=3))
print(confusion_matrix(feat["is_anomaly"], feat["flag_hybrid"]))

              precision    recall  f1-score   support

           0      0.998     0.994     0.996     19741
           1      0.669     0.846     0.747       299

    accuracy                          0.991     20040
   macro avg      0.833     0.920     0.872     20040
weighted avg      0.993     0.991     0.992     20040

[[19616   125]
 [   46   253]]


In [13]:
feat[feat["is_anomaly"] == 1].groupby("anomaly_type")["flag_hybrid"].mean().round(2)

anomaly_type
duplicate    1.00
odd_hour     1.00
spike        0.69
Name: flag_hybrid, dtype: float64

In [14]:
rows = []
for cont in [0.005, 0.01, 0.015, 0.02, 0.03]:
    m = IsolationForest(n_estimators=200, contamination=cont, random_state=42).fit(feat[ML_FEATURES])
    flag = ((m.predict(feat[ML_FEATURES]) == -1) | (feat["is_repeat"] == 1)).astype(int)
    tp = ((flag == 1) & (feat["is_anomaly"] == 1)).sum()
    rows.append({
        "contamination": cont,
        "flagged": flag.sum(),
        "precision": round(tp / flag.sum(), 3),
        "recall": round(tp / feat["is_anomaly"].sum(), 3),
    })
pd.DataFrame(rows)

,contamination,flagged,precision,recall
0,0.005,178,0.966,0.575
1,0.010,278,0.838,0.779
2,0.015,378,0.669,0.846
3,0.020,478,0.554,0.886
4,0.030,679,0.409,0.930


In [15]:
missed = feat[(feat["anomaly_type"] == "spike") & (feat["flag_hybrid"] == 0)]
print(len(missed), "spikes missed")
print(missed["category"].value_counts())
print(missed[["category", "amount", "amount_ratio", "log_ratio"]].describe().round(2))

46 spikes missed
category
Transport            18
Food                  9
Groceries             6
Bills & Utilities     3
Shopping              3
Health                2
Other                 2
Travel                2
Entertainment         1
Name: count, dtype: int64
         amount  amount_ratio  log_ratio
count     46.00         46.00      46.00
mean    3518.89          5.22       1.61
std     4475.42          1.49       0.31
min      393.00          2.17       0.77
25%     1147.75          4.21       1.44
50%     1668.50          5.18       1.64
75%     4683.25          6.32       1.84
max    23472.00          7.84       2.06
